# 🎯 [DAT10_M09] AAR 기반 동기화 누락 및 데이터 충돌 사례 분석 및 동기화 룰셋 재튜닝
### Google Colab 실습 환경 (데이터·자동화·업무플랫폼 / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/DAT10/dat10_m09_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 데이터·자동화·업무플랫폼 (Level 3) |
| **미션 ID** | DAT10_M09 |
| **미션 주제** | AAR 기반 동기화 누락 및 데이터 충돌 사례 분석 및 동기화 룰셋 재튜닝 |
| **소요 시간** | 약 3.0시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 배경 및 당위성
전시에는 전술 통신망이 단절된 채 예하 보급소가 입고와 불출을 각자 기록하고, 통신 복구 후 기록을 병합하는 과정에서 중복·누락·충돌·위변조가 빈번히 발생하며, 이를 적시에 통제하지 못하면 재고 왜곡과 방산 공급망 위기 식별이 지연됩니다. 본 프로젝트는 방산 공급망 회복력 연구(미국 방위산업기반 정책연구)의 문제의식을 교육용 가상 자료로 재구성해, 가상 보급소 5곳이 통신 두절 환경에서 오프라인으로 재고를 기록하고 동기화하는 SaaS를 개발하고, 동기화된 데이터에서 공급망 재난 징후(업체 공급 감소, 재고 소진 임박)를 식별하는 End-to-End 군수 소프트웨어 엔지니어링 과제입니다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 09. AAR 기반 동기화 누락 및 데이터 충돌 사례 분석 및 동기화 룰셋 재튜닝
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 기반 동기화 누락 및 데이터 충돌 사례를 분석하고 동기화 룰을 재튜닝한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **동기화 누락·충돌 판정 오류 사례를 AAR로 원인별로 분류하고 규칙 파라미터(중복 판정 방식, 충돌 탐지 규칙, 자동 병합 임계, 재시도 횟수)를 재튜닝해 종합 정확도 10%p 이상 향상과 재발 오류 0건을 시험한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **가상 사례 1,200건(중복 재전송·재입력, 같은 크기의 서로 다른 거래, 동시·후속 메타 수정과 시계 어긋남, 수량 차이, 통신불량 전송 실패)에 현행 규칙을 적용해 오류를 D1~D6 원인으로 분류 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/dat10_aar_report.md 및 configs/dat10_tuned_params.yaml` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 오류 원인 규명의 객관성 및 개선 조치의 논리적 타당성

#### 미션 PASS 조건

- 정량 기준: AAR 피드백 적용 후 종합 정확도 10%p 이상 향상, 재발 오류 0건


- 진행 기준: AAR 보고서 작성 및 파라미터 튜닝 전후 회귀 테스트 통과


- 검증 방법: 회귀 테스트(Regression Test) 스위트 전체 자동 실행 및 비교


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""동기화 사후 검토(AAR) 사례 1,200건에서 판정 오류를 원인(중복 판정 방식·시계 어긋남·충돌 탐지 규칙·자동 병합 임계·재시도 횟수)별로 분류하고 규칙 파라미터를 재튜닝해, 별도 시험 3,000건의 종합 정확도 향상폭·AAR 사례 재발 오류·회귀 테스트 통과를 계산하고 튜닝 결과를 YAML 문자열로 출력한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

KINDS = ["dup_resend", "distinct_similar", "meta_conflict", "meta_causal", "qty_diff", "send_fail", "dup_reinput"]
PK = [.20, .12, .18, .15, .15, .15, .05]
CAUSE = {"dup_resend": "D2 재전송 중복 놓침", "distinct_similar": "D1 다른 거래를 중복으로 판정", "meta_conflict": "D4 동시 수정 놓침(조용한 덮어쓰기)", "meta_causal": "D3 후속 수정의 승자 오판·불필요 경보",
         "qty_diff": "D5 자동 병합 임계 부적절", "send_fail": "D6 재시도 부족 누락", "dup_reinput": "D2b 재입력(새 ID) 중복 놓침"}


def make_events(seed, n):
    """[교육용가상자료] 동기화 사례 생성. 단말 시계 오차(+-2시간), 버전 벡터 없는 구형 기록 8%, 통신불량 상황의 전송 실패를 포함한다."""
    rng = np.random.default_rng(seed); out = []
    for i in range(n):
        k = str(rng.choice(KINDS, p=PK)); o = rng.uniform(-2, 2, 2); ev = {"id": i, "kind": k, "vv": rng.random() > 0.08}
        if k == "dup_resend":
            ev.update(same_id=True, dt=0.0 if rng.random() < .5 else float(rng.uniform(0.5, 6)))
        elif k == "distinct_similar":
            ev.update(same_id=False, dt=float(rng.uniform(0, 0.5)))
        elif k == "dup_reinput":                                           # 오입력 정정 등으로 같은 내용을 새 ID로 다시 입력
            ev.update(same_id=False, dt=float(rng.uniform(0.1, 6)))
        elif k in ("meta_conflict", "meta_causal"):
            gap = float(rng.uniform(0.5, 60)) if k == "meta_conflict" else float(rng.uniform(0.01, 1.5))
            ev.update(ta=100 + o[0], tb=100 + gap + o[1])                  # 기록된 시각 = 실제 시각 + 단말 시계 오차
        elif k == "qty_diff":
            ev.update(d=float(rng.uniform(0.0, 0.8)))
        else:
            ev.update(p=0.5 if rng.random() < .6 else 0.8, u=rng.random(8))
        out.append(ev)
    return out


def decide(ev, P):
    k = ev["kind"]
    if k in ("dup_resend", "distinct_similar", "dup_reinput"):
        if P["dup"] == "id":
            return "skip" if ev["same_id"] else "apply"
        if P["dup"] == "id+content":
            return "skip" if ev["same_id"] or ev["dt"] <= P["win"] else "apply"
        return "skip" if ev["dt"] <= P["win"] else "apply"                  # 내용 비교: 같은 품목·수량이 시간창 안이면 중복
    if k in ("meta_conflict", "meta_causal"):
        if P["conflict"] == "vv" and ev["vv"]:
            return "hitl" if k == "meta_conflict" else "pick_b"
        if abs(ev["ta"] - ev["tb"]) <= P["tw"]:
            return "hitl"
        return "pick_b" if ev["tb"] > ev["ta"] else "pick_a"                # 마지막 기록 우선(LWW)
    if k == "qty_diff":
        return "auto" if ev["d"] < P["thr"] else "hitl"
    return "applied" if (ev["u"][:1 + P["retries"]] < ev["p"]).any() else "failed"


def truth(ev):
    k = ev["kind"]
    if k == "qty_diff":
        return "auto" if ev["d"] < 0.25 else "hitl"
    return {"dup_resend": "skip", "dup_reinput": "skip", "distinct_similar": "apply", "meta_conflict": "hitl", "meta_causal": "pick_b", "send_fail": "applied"}[k]


def evaluate(evs, P):
    ok = [decide(e, P) == truth(e) for e in evs]
    return np.mean(ok), pd.Series([CAUSE[e["kind"]] for e, o in zip(evs, ok) if not o]).value_counts(), ok


P0 = {"dup": "content", "win": 2.0, "conflict": "time", "tw": 4.0, "thr": 0.3, "retries": 2}           # 초기 운용 규칙(M3 단계 설정)
aar, test = make_events(31, 1200), make_events(77, 3000)
a0, causes, ok0 = evaluate(aar, P0)
print(f"[AAR 사례] {len(aar)}건, 현행 규칙 정확도 {a0:.1%}, 오류 {int(len(aar) * (1 - a0))}건")
print("[원인 분류] 원인 / 건수 / 비율")
for c, n in causes.items():
    print(f"  {c:28s} {n:4d}건 {n / (len(aar) * (1 - a0)):5.1%}")

# ---- 재튜닝: 사례 종류가 서로 다르므로 파라미터마다 AAR 사례에서 따로 탐색(재시도는 횟수당 지연 비용 0.2%p 가산) ----
def best(param, grid, kinds, cost=lambda v: 0.0):
    sub = [e for e in aar if e["kind"] in kinds]; sc = []
    for v in grid:
        P = {**P0, **(v if isinstance(v, dict) else {param: v})}; sc.append((np.mean([decide(e, P) == truth(e) for e in sub]) - cost(v), v))
    return max(sc, key=lambda x: x[0]), sc
(_, dup_best), _ = best("dup", [{"dup": "content", "win": w} for w in (0.0, 0.25, 0.5, 1, 2, 4)] + [{"dup": "id+content", "win": w} for w in (0.25, 0.5, 1, 2, 4)] + [{"dup": "id"}], ("dup_resend", "distinct_similar", "dup_reinput"))
(_, cf_best), _ = best("conflict", [{"conflict": "time", "tw": w} for w in (0.25, 0.5, 1, 2, 4)] + [{"conflict": "vv", "tw": w} for w in (0.5, 1, 2, 4, 8)], ("meta_conflict", "meta_causal"))
(_, thr_best), _ = best("thr", [0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5], ("qty_diff",))
(_, rt_best), _ = best("retries", [1, 2, 3, 4, 5, 6], ("send_fail",), cost=lambda v: 0.002 * v)
P1 = {**P0, **dup_best, **cf_best, "thr": thr_best, "retries": rt_best}

a_aar, rec, ok1 = evaluate(aar, P1)
a_old_t, c_old_t, _ = evaluate(test, P0)
a_new_t, c_new_t, _ = evaluate(test, P1)
rec_n = int(sum((not o0) and (not o1) for o0, o1 in zip(ok0, ok1)))
print(f"\n[재튜닝 결과] 중복 판정 {P0['dup']}(창 {P0['win']}h) -> {P1['dup']}{'' if P1['dup'] == 'id' else '(창 ' + str(P1['win']) + 'h)'}, 충돌 탐지 {P0['conflict']}(창 {P0['tw']}h) -> {P1['conflict']}(구형 기록은 시간창 {P1['tw']}h), 자동 병합 임계 {P0['thr']} -> {P1['thr']}, 재시도 {P0['retries']} -> {P1['retries']}회")
print(f"AAR 사례 재실행: 정확도 {a0:.1%} -> {a_aar:.1%}, 같은 사례에서 오류 재발 {rec_n}건 (목표 0건: {'달성' if rec_n == 0 else '미달'})")
print(f"별도 시험 3,000건: {a_old_t:.1%} -> {a_new_t:.1%} ({(a_new_t - a_old_t) * 100:+.1f}%p, 목표 +10%p 이상: {'달성' if a_new_t - a_old_t >= .10 else '미달'})")
print("시험 3,000건 남은 오류:", {c: int(n) for c, n in c_new_t.items()} or "없음")
tt = pd.DataFrame({"before": c_old_t, "after": c_new_t}).fillna(0).astype(int)

# ---- 회귀 테스트: 규칙 단위 시험 8개를 현행/재튜닝 규칙에 각각 실행 ----
T = [("같은 ID 재전송(시각 3h 차이)은 건너뜀", {"kind": "dup_resend", "same_id": True, "dt": 3.0, "vv": True}, "skip"), ("같은 ID 재전송(시각 동일)은 건너뜀", {"kind": "dup_resend", "same_id": True, "dt": 0.0, "vv": True}, "skip"),
     ("다른 ID의 같은 수량 거래는 반영", {"kind": "distinct_similar", "same_id": False, "dt": 0.2, "vv": True}, "apply"), ("버전 벡터 동시 수정은 군수관 검토", {"kind": "meta_conflict", "ta": 100, "tb": 130, "vv": True}, "hitl"),
     ("버전 벡터 선후 관계는 후속 값 채택(시계 역전)", {"kind": "meta_causal", "ta": 101.0, "tb": 100.5, "vv": True}, "pick_b"), ("차이 10%는 자동 병합", {"kind": "qty_diff", "d": 0.10, "vv": True}, "auto"),
     ("차이 30%는 군수관 검토", {"kind": "qty_diff", "d": 0.30, "vv": True}, "hitl"), ("통신불량 전송 3회 안에 성공", {"kind": "send_fail", "p": 0.5, "u": np.array([.9, .8, .1, 0, 0, 0, 0, 0]), "vv": True}, "applied")]
print("\n[회귀 테스트 8개] 현행 / 재튜닝")
p0 = [decide(e, P0) == t for _, e, t in T]; p1 = [decide(e, P1) == t for _, e, t in T]
for (nm, _, _), x, y in zip(T, p0, p1):
    print(f"  {nm:34s} {'통과' if x else '실패'} / {'통과' if y else '실패'}")
regress = sum(a and not b for a, b in zip(p0, p1))
print(f"통과 {sum(p0)}/8 -> {sum(p1)}/8, 기존 통과 항목이 깨진 회귀 {regress}건 ({'전체 통과' if all(p1) else '실패 있음'})")

# ---- 산출물: 튜닝 파라미터 YAML 문자열(configs/dat10_tuned_params.yaml 내용) ----
yml = "\n".join(["sync_rules:", f"  dedup_key: {P1['dup']}", f"  dedup_window_h: {P1['win'] if P1['dup'] != 'id' else 0}", f"  conflict_detect: {P1['conflict']}", f"  legacy_time_window_h: {P1['tw']}", f"  auto_merge_diff_max: {P1['thr']}", f"  send_retries: {P1['retries']}", "  hitl_required_when: [conflict, diff_ge_0.25]"])
print("\n[configs/dat10_tuned_params.yaml]"); print(yml)
print("[지속적 학습 연계] 이번 AAR 오류 유형을 사례 은행에 누적 -> 월 1회 새 사례로 파라미터 재탐색 -> 회귀 테스트 8개 통과 시에만 후보 반영 -> 군수관 승인 후 배포(자동 배포 금지)")
print("※ 사례는 가상 생성이며 시험 분포가 튜닝 분포와 같아 향상폭이 실제보다 낙관적이다. 규칙 변경은 군수관 승인과 회귀 테스트를 거친다.")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].bar(["before", "after"], [a_old_t * 100, a_new_t * 100], color=["tab:gray", "tab:blue"]); ax[0].set_ylim(50, 100); ax[0].set_ylabel("accuracy (%)"); ax[0].set_title("Test accuracy (3,000 cases)")
x = np.arange(len(tt)); ax[1].bar(x - .2, tt["before"], .4, label="before", color="tab:gray"); ax[1].bar(x + .2, tt["after"], .4, label="after", color="tab:blue")
ax[1].set_xticks(x); ax[1].set_xticklabels([c.split()[0] for c in tt.index]); ax[1].set_title("Errors by cause"); ax[1].legend(); plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR 사례] 1200건, 현행 규칙 정확도 45.5%, 오류 653건
[원인 분류] 원인 / 건수 / 비율
  D4 동시 수정 놓침(조용한 덮어쓰기)         193건 29.5%
  D3 후속 수정의 승자 오판·불필요 경보        177건 27.1%
  D1 다른 거래를 중복으로 판정             139건 21.3%
  D2 재전송 중복 놓침                   95건 14.5%
  D2b 재입력(새 ID) 중복 놓침            25건  3.8%
  D6 재시도 부족 누락                   17건  2.6%
  D5 자동 병합 임계 부적절                 8건  1.2%

[재튜닝 결과] 중복 판정 content(창 2.0h) -> id, 충돌 탐지 time(창 4.0h) -> vv(구형 기록은 시간창 0.5h), 자동 병합 임계 0.3 -> 0.25, 재시도 2 -> 6회
AAR 사례 재실행: 정확도 45.5% -> 93.7%, 같은 사례에서 오류 재발 49건 (목표 0건: 미달)
별도 시험 3,000건: 44.7% -> 93.0% (+48.3%p, 목표 +10%p 이상: 달성)
시험 3,000건 남은 오류: {'D2b 재입력(새 ID) 중복 놓침': 164, 'D4 동시 수정 놓침(조용한 덮어쓰기)': 32, 'D3 후속 수정의 승자 오판·불필요 경보': 11, 'D6 재시도 부족 누락': 2}

[회귀 테스트 8개] 현행 / 재튜닝
  같은 ID 재전송(시각 3h 차이)은 건너뜀           실패 / 통과
  같은 ID 재전송(시각 동일)은 건너뜀              통과 / 통과
  다른 ID의 같은 수량 거래는 반영                실패 / 통과
  버전 벡터 동시 수정은 군수관 검토                실패 / 통과
  버전 벡터 선후 관계는 후속 값 채택(시계 역전)        실패 / 통과
  차이 10%는 자동 병합                      통과 / 통과
  차이 30%는 군수관 검토                     통과 / 통과
  통신불량 전송 3회 안에 성공                   통과 / 통과
통과 4/8 -> 8/8, 기존 통과 항목이 깨진 회귀 0건 (전체 통과)

[configs/dat10_tuned_params.yaml]
sync_rules:
  dedup_key: id
  dedup_window_h: 0
  conflict_detect: vv
  legacy_time_window_h: 0.5
  auto_merge_diff_max: 0.25
  send_retries: 6
  hitl_required_when: [conflict, diff_ge_0.25]
[지속적 학습 연계] 이번 AAR 오류 유형을 사례 은행에 누적 -> 월 1회 새 사례로 파라미터 재탐색 -> 회귀 테스트 8개 통과 시에만 후보 반영 -> 군수관 승인 후 배포(자동 배포 금지)
※ 사례는 가상 생성이며 시험 분포가 튜닝 분포와 같아 향상폭이 실제보다 낙관적이다. 규칙 변경은 군수관 승인과 회귀 테스트를 거친다.
```
</details>
